# Building Models

In PyTorch you subclass `nn.Module` and define `forward()`. In idris-ml, layers
implement the `LayerLike` interface and compose into `Network` chains using `~>`.

The compiler checks that each layer's output dimension matches the next layer's input.

## Creating Layers

Layers with learnable parameters (linear, RNN, etc.) return `IO` because weight
initialization uses random numbers. Stateless layers (relu, softmax) are pure values.

In [1]:
:t linearLayer

Layer.Linear.linearLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [2]:
:t reluLayer

Layer.Activation.reluLayer : (Ord ty, FromDouble ty) => AnyLayer n n ty


Notice: `linearLayer` returns `IO (AnyLayer i o ty)` while `reluLayer` is
just `AnyLayer n n ty` — no IO, no parameters, input and output have the same size.

## Composing Networks

`~>` chains layers, `OutputLayer` terminates the chain.
`autoName` assigns parameter names for gradient tracking.

In [3]:
:exec do { ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (ll ~> OutputLayer softmaxLayer));
  putStrLn (show model) }

Error: Can't find an implementation for Show (Network 2 [3] 3 (Variable ?d)).

(Interactive):1:111--1:121
 1 | :exec do { ll <- linearLayer {i=2, o=3}; model <- pure (autoName (ll ~> OutputLayer softmaxLayer)); putStrLn (show model) }
                                                                                                                   ^^^^^^^^^^


This is equivalent to PyTorch's:
```python
nn.Sequential(nn.Linear(2, 3), nn.Softmax(dim=-1))
```

But the compiler verifies that all dimensions chain correctly.

In [4]:
:exec do { l1 <- linearLayer {i=2, o=8};
  l2 <- linearLayer {i=8, o=3};
  model <- pure (autoName (l1 ~> reluLayer ~> l2 ~> OutputLayer softmaxLayer));
  putStrLn (show model) }

Error: Can't find an implementation for Show (Network 2 [8, 8, 3] 3 (Variable ?d)).

(Interactive):1:160--1:170
 1 | :exec do { l1 <- linearLayer {i=2, o=8}; l2 <- linearLayer {i=8, o=3}; model <- pure (autoName (l1 ~> reluLayer ~> l2 ~> OutputLayer softmaxLayer)); putStrLn (show model) }
                                                                                                                                                                    ^^^^^^^^^^


## Dimension Mismatch = Compile Error

What happens when output 8 doesn't match input 5?

In [5]:
:exec do { l1 <- linearLayer {i=4, o=8};
  l2 <- linearLayer {i=5, o=3};
  model <- pure (autoName (l1 ~> l2 ~> OutputLayer softmaxLayer));
  putStrLn "should not reach here" }

Error: When unifying:
    Network 5 [3] 3 ?ty
and:
    Network 8 [3] 3 ?ty
Mismatch between: 0 and 3.

(Interactive):1:103--1:133
 1 | :exec do { l1 <- linearLayer {i=4, o=8}; l2 <- linearLayer {i=5, o=3}; model <- pure (autoName (l1 ~> l2 ~> OutputLayer softmaxLayer)); putStrLn "should not reach here" }
                                                                                                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


The compiler catches the mismatch between output dimension 8 and input dimension 5.
In PyTorch this would be a `RuntimeError` during the first forward pass — here it
never gets that far.

## Why `autoName` Matters

Every learnable parameter needs a unique name so the optimizer can find and update it.
`autoName` assigns names like `ll0_weight0`, `ll0_bias0`, etc.
Without it, parameters are invisible to the gradient system and training does nothing.

## Forward Pass

`forward` runs input through the model and returns both the **updated model** and
the output. The model is returned because layers can have mutable state (RNN hidden
state, batch norm running stats).

For evaluation (no gradients), convert to a `Double` model with `toDoubleNetwork`:

In [6]:
:t forward

Layer.Core.forward : (FromDouble ty, (Floating ty, (Fractional ty, (Neg ty, (Num ty, Ord ty))))) => Network i hs o ty -> Vector i ty -> (Network i hs o ty, Vector o ty)


In [7]:
:exec do { srand 42;
  ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (OutputLayer ll));
  dblModel <- pure (toDoubleNetwork model);
  result <- pure (forward dblModel (the (Vector 2 Double) (VTensor [1.0, 2.0])));
  putStrLn ("Output: " ++ show (snd result)) }

Error: Can't find an implementation for (Num ?ty, FromDouble ?ty).

(Interactive):1:28--1:50
 1 | :exec do { srand 42; ll <- linearLayer {i=2, o=3}; model <- pure (autoName (OutputLayer ll)); dblModel <- pure (toDoubleNetwork model); result <- pure (forward dblModel (the (Vector 2 Double) (VTensor [1.0, 2.0]))); putStrLn ("Output: " ++ show (snd result)) }
                                ^^^^^^^^^^^^^^^^^^^^^^


Compare to PyTorch where `model(x)` returns only the output and mutates
the model in-place. In idris-ml, nothing is mutated — you get a new model back.

## Available Layers

Discover what's available using `:t`. These are the main layer constructors:

In [8]:
:t rnnLayer

Layer.Rnn.rnnLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [9]:
:t lstmLayer

Layer.Lstm.lstmLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [10]:
:t gruLayer

Layer.Gru.gruLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [11]:
:t conv2dLayer

Layer.Conv.conv2dLayer : (Num ty, FromDouble ty) => IO (AnyLayer (inC * (h * w)) (outC * (ConvOutDim h kH padH * ConvOutDim w kW padW)) ty)


In [12]:
:t dropoutLayer

Layer.Dropout.dropoutLayer : Double -> AnyLayer n n ty


In [13]:
:t embeddingLayer

Layer.Embedding.embeddingLayer : (Num ty, FromDouble ty) => IO (AnyLayer seqLen (seqLen * embedDim) ty)


Activation and normalization layers have no learnable parameters:

In [14]:
:t softmaxLayer

Layer.Normalization.softmaxLayer : (Fractional ty, Floating ty) => AnyLayer n n ty


In [15]:
:t tanhLayer

Layer.Activation.tanhLayer : (FromDouble ty, (Neg ty, (Fractional ty, Floating ty))) => AnyLayer n n ty


In [16]:
:t sigmoidLayer

Layer.Activation.sigmoidLayer : (FromDouble ty, (Neg ty, (Fractional ty, Floating ty))) => AnyLayer n n ty


For a full list of everything in any module, use `:browse`:

In [17]:
:browse Layer.Linear

LinearState : Nat -> Nat -> Type -> Type
MkLinear : Matrix outputSize inputSize ty -> Vector outputSize ty -> Maybe AnyPtr -> Maybe AnyPtr -> LinearState inputSize outputSize ty
bias : LinearState inputSize outputSize ty -> Vector outputSize ty
biasTensor : LinearState inputSize outputSize ty -> Maybe AnyPtr
buildViewMatrix : String -> AnyPtr -> Int -> Int -> (rows : Nat) -> (cols : Nat) -> Vect rows (Vector cols (Variable d))
buildViewRow : String -> AnyPtr -> Int -> Int -> Int -> (k : Nat) -> Vect k (Scalar (Variable d))
buildViewVector : String -> AnyPtr -> Int -> (k : Nat) -> Vect k (Scalar (Variable d))
extractBiasTensor : LinearState i o (Variable d) -> Maybe AnyPtr
extractWeightTensor : LinearState i o (Variable d) -> Maybe AnyPtr
linearLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)
linearLayerWith : (Num ty, FromDouble ty) => InitStrategy -> IO (AnyLayer i o ty)
linearLayerWithBias : (Num ty, FromDouble ty) => InitStrategy -> Double -> IO (AnyLayer i o ty)
mkLinear : (

Next: [03 Data and Loss](03_data_and_loss.ipynb) — creating training data
and computing loss with type-checked dimensions.